### Stadtführung – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023), 1. Runde, Aufgabe 5*

Die Tour eines Stadtführers besucht Orte in chronologischer Reihenfolge; ein **Tourpunkt** ist die Kombination aus einem Ort und
einem Jahr, ein Ort kommt oft mehrfach vor. Seine Tochter Alina will die Tour straffen. Einige Tourpunkte sind **essentiell**,
sie müssen bleiben. Gekürzt wird nur durch Weglassen **geschlossener Teiltouren**: Hat der Vater das Rathaus 1678 besucht, danach einen
Schlenker durch die Altstadt gemacht und das Rathaus 1739 noch einmal, kann Alina auf den Schlenker verzichten und am Rathaus
von beiden Jahren erzählen. Die neue Tour soll wie die alte am Ort ihres Beginns enden, der Startort darf aber ein anderer sein.

#### Aufgabe 5

Finde eine kürzeste Teiltour der Tour des Vaters, die chronologisch geordnet bleibt und alle essentiellen Tourpunkte besucht. Das
Programm soll auch funktionieren, wenn Alina die essentiellen Tourpunkte später ändert. Wende es auf alle Beispiele von den
BWINF-Webseiten an und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/tour1.txt` bis `tour5.txt`)

- 1. Zeile: Anzahl `n` der Tourpunkte
- folgende `n` Zeilen: `Ort,Jahr,Essentiell,kumulierter Abstand`, durch Kommas getrennt; an dritter Stelle steht `X` (essentiell)
  oder ein Leerzeichen
- Der kumulierte Abstand ist die Weglänge vom Start der Tour bis zu diesem Tourpunkt. Der Abstand zwischen zwei aufeinanderfolgenden
  Tourpunkten ist die Differenz (Brauerei → Karzer: 80, Karzer → Rathaus: 150 − 80 = 70).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Aufgabe genau verstehen:** Man darf nur *geschlossene* Teiltouren streichen (Anfang und Ende am gleichen Ort), und darin darf
  kein essentieller Tourpunkt liegen. Reihenfolge ändern ist nicht erlaubt: Einen essentiellen Punkt auszulassen, um ihn später
  nachzuholen, geht nicht.
- **Zerlegen in Teilprobleme:** Die essentiellen Tourpunkte bleiben alle erhalten. Also kann man jedes Stück zwischen zwei
  aufeinanderfolgenden essentiellen Punkten **getrennt** kürzen und die Längen addieren.
- **Dynamische Programmierung (Kürzester Weg):** Für jeden Tourpunkt `i` merken wir uns die kürzeste Strecke vom Anfang des Stücks
  bis `i`. Zu `i` kommt man auf höchstens zwei Arten: von `i - 1` aus (Weg der Tour weitergehen) oder, falls der Ort schon vorher
  besucht wurde, von dort aus ohne Umweg (Schlenker weglassen). Es zählt die kürzere.
- **Wörterbuch (`dict`):** Es merkt sich zu jedem Ort den letzten Tourpunkt.
- **Ring:** Wenn der Startort wechseln darf, behandeln wir das Ende der Tour und den Anfang wie *zusammengeklebt*.

**Nützliche Funktionen und Bibliotheken**
- `split(',')` und `strip()`: Zeile in Felder zerlegen, Leerzeichen entfernen
- `zip(liste, liste[1:])`: aufeinanderfolgende Paare einer Liste
- `dict`: `ort in letzter` fragt, ob ein Ort schon vorkam

## Die Idee

**Warum kein einfaches Greedy-Verfahren?** Man könnte immer den größten möglichen Schlenker streichen. Das ist nicht immer optimal:
Es kann zwei kleinere Schlenker geben, die zusammen mehr sparen als der eine große, der sie beide überlappt. Wir brauchen ein Verfahren,
das keine Fälle übersieht.

**Teilproblem:** Gegeben seien zwei aufeinanderfolgende essentielle Tourpunkte `a` und `b`. Dazwischen liegen nur nicht essentielle
Punkte. Wie kurz kann die Strecke von `a` nach `b` werden?

Sei `dist[i]` die kürzeste Strecke von `a` bis zum Tourpunkt `i` (`a <= i <= b`). Dann gilt:

- `dist[a] = 0`
- Normalerweise: `dist[i] = dist[i - 1] + Abstand(i - 1, i)`
- War der Ort von `i` schon vorher bei einem Tourpunkt `j` im Stück (der letzte davor), kann man den Schlenker von `j` bis `i` weglassen.
  Dann ist `dist[i] = dist[j]`, falls das kürzer ist.

Der Vorgänger `j` genügt: Gab es noch frühere Besuche des Ortes, ergeben sie sich durch mehrfaches Anwenden dieser Regel. Ein
essentieller Punkt liegt nie *zwischen* `a` und `b`, deshalb darf jeder Schlenker innerhalb des Stücks gestrichen werden.

**Der Startort:** Dürfen wir einen neuen Startort wählen, kann auch vor dem ersten und nach dem letzten essentiellen Punkt gekürzt werden.
Wir kleben das Ende der Tour an den Anfang (der Vater endet am Startort) und suchen den kürzesten Weg vom **letzten** zum **ersten**
essentiellen Punkt *um den Ring herum*. Wählt der Weg eine Abkürzung über die Klebestelle, entsteht ein neuer Start.

## Schritt 1: Einlesen

Jede Zeile hat vier durch Kommas getrennte Felder. Das dritte ist `X` oder ein Leerzeichen; mit `strip()` entfernen wir das
Leerzeichen (und Leerzeichen am Zeilenende, wie in `tour4.txt`). Wir speichern vier Listen: `orte`, `jahre`, `essentiell` (`True` oder
`False`) und `kumuliert`.

In [ ]:
def lies_tour(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    n = int(zeilen[0])
    orte, jahre, essentiell, kumuliert = [], [], [], []
    for zeile in zeilen[1:1 + n]:
        felder = [feld.strip() for feld in zeile.split(',')]
        orte.append(felder[0])
        jahre.append(int(felder[1]))
        essentiell.append(felder[2] == 'X')
        kumuliert.append(int(felder[3]))
    return orte, jahre, essentiell, kumuliert


orte, jahre, essentiell, kumuliert = lies_tour('beispieldaten/tour1.txt')
print(len(orte), 'Tourpunkte')
for i in range(5):
    print(orte[i], jahre[i], essentiell[i], kumuliert[i])

18 Tourpunkte
Brauerei 1613 True 0
Karzer 1665 True 80
Rathaus 1678 True 150
Gründungsstein 1685 False 310
Wallanlage 1690 False 410


## Schritt 2: Abstände

Der Abstand von Tourpunkt `i - 1` nach `i` ist die Differenz der kumulierten Abstände. Für `i = 0` gibt es keinen Vorgänger, dort
steht 0.

In [ ]:
def abstaende_berechnen(kumuliert):
    abstaende = [0]
    for i in range(1, len(kumuliert)):
        abstaende.append(kumuliert[i] - kumuliert[i - 1])
    return abstaende


abstaende = abstaende_berechnen(kumuliert)
print(abstaende)

[0, 80, 70, 160, 100, 90, 180, 30, 120, 130, 130, 90, 120, 100, 50, 330, 130, 150]


## Schritt 3: Kürzester Weg in einem Stück

`kuerzester_weg(orte_stueck, abstaende_stueck)` bekommt die Orte eines Stücks und die Abstände (Eintrag 0 wird nicht benutzt) und
berechnet nach der Idee oben `dist[i]`. Zusätzlich speichern wir in `vorgaenger[i]`, von welchem Tourpunkt wir gekommen sind (`i - 1`
oder der frühere Besuch desselben Ortes). Am Ende gehen wir von hinten die Vorgänger entlang und erhalten die Punkte, die in der
gekürzten Tour bleiben.

Wichtig: Bei Gleichstand nehmen wir den normalen Weg (`>` statt `>=`). Dann bleiben möglichst viele Tourpunkte erhalten, ohne dass
die Strecke länger wird.

In [ ]:
def kuerzester_weg(orte_stueck, abstaende_stueck):
    """Gibt (Länge, Liste der Positionen im Stück, die in der gekürzten Tour bleiben) zurück."""
    anzahl = len(orte_stueck)
    dist = [0] * anzahl
    vorgaenger = [None] * anzahl
    letzter = {}                                # Ort -> letzter Tourpunkt an diesem Ort
    for i in range(anzahl):
        if i > 0:
            dist[i] = dist[i - 1] + abstaende_stueck[i]       # Tour einfach weitergehen
            vorgaenger[i] = i - 1
            ort = orte_stueck[i]
            if ort in letzter and dist[i] > dist[letzter[ort]]:
                dist[i] = dist[letzter[ort]]                  # Schlenker weglassen
                vorgaenger[i] = letzter[ort]
        letzter[orte_stueck[i]] = i
    # Von hinten die Vorgänger entlanggehen
    bleibt = [anzahl - 1]
    while vorgaenger[bleibt[-1]] is not None:
        bleibt.append(vorgaenger[bleibt[-1]])
    bleibt.reverse()
    return dist[anzahl - 1], bleibt


# Das Stück zwischen den ersten beiden essentiellen Punkten von tour1: Brauerei 1613 bis Rathaus 1678 (keine Wiederholung)
print(kuerzester_weg(orte[0:3], abstaende[0:3]))
# Von Rathaus 1678 (Index 2) bis Rathaus 1739 (Index 5): der Schlenker wird ganz gestrichen
print(orte[2:6])
print(kuerzester_weg(orte[2:6], abstaende[2:6]))

(150, [0, 1, 2])
['Rathaus', 'Gründungsstein', 'Wallanlage', 'Rathaus']
(0, [0, 3])


## Schritt 4: Die ganze Tour kürzen

`kuerze_tour(orte, essentiell, abstaende, startort_frei)` liefert die Länge und die Indizes der Tourpunkte der neuen Tour.

1. `E` ist die Liste der Indizes der essentiellen Tourpunkte. Gibt es höchstens einen, ist die kürzeste Tour dieser eine Punkt
   (Länge 0). Gibt es keinen, nehmen wir irgendeinen Punkt.
2. **Mitte:** Für jedes Paar aufeinanderfolgender essentieller Punkte `(a, b)` rufen wir `kuerzester_weg` mit dem Stück `a` bis `b`
   auf und hängen die bleibenden Punkte (umgerechnet in Indizes der ganzen Tour) an.
3. **Anfang und Ende, Startort fest:** Ist der Startort nicht frei, kürzen wir das Stück von `0` bis zum ersten und das Stück
   vom letzten essentiellen Punkt bis `n - 1` ebenfalls einzeln.
4. **Anfang und Ende, Startort frei:** Wir bilden den **Ring** `ring = [letzter essentieller Punkt, ..., n - 1, 0, ..., erster essentieller Punkt]`.
   Der Abstand von `n - 1` nach `0` ist 0, denn der Vater endet dort, wo er beginnt (beide Tourpunkte liegen am gleichen Ort). `kuerzester_weg`
   findet jetzt auch Abkürzungen über die Klebestelle. Die Punkte des Rings mit Index kleiner oder gleich dem ersten
   essentiellen Punkt gehören zum neuen Anfang, die übrigen zum neuen Ende.

In [ ]:
def kuerze_tour(orte, essentiell, abstaende, startort_frei):
    n = len(orte)
    E = [i for i in range(n) if essentiell[i]]
    if len(E) <= 1:
        return 0, E if E else [0]
    erster, letzter = E[0], E[-1]

    # Mitte: je zwei aufeinanderfolgende essentielle Punkte getrennt kürzen
    laenge = 0
    mitte = [erster]
    for a, b in zip(E, E[1:]):
        indizes = list(range(a, b + 1))
        l, bleibt = kuerzester_weg([orte[i] for i in indizes], [abstaende[i] for i in indizes])
        laenge += l
        for pos in bleibt[1:]:                  # der Punkt a ist schon in der Liste
            mitte.append(indizes[pos])

    if startort_frei:
        ring = list(range(letzter, n)) + list(range(0, erster + 1))
        ring_abstaende = [0]
        for i in ring[1:]:
            ring_abstaende.append(abstaende[i] if i != 0 else 0)   # Klebestelle n-1 -> 0 kostet nichts
        l, bleibt = kuerzester_weg([orte[i] for i in ring], ring_abstaende)
        laenge += l
        anfang = [ring[pos] for pos in bleibt if ring[pos] <= erster]
        ende = [ring[pos] for pos in bleibt if ring[pos] >= letzter]
        return laenge, anfang[:-1] + mitte + ende[1:]

    # Startort fest: Anfang und Ende einzeln kürzen
    indizes = list(range(0, erster + 1))
    l, bleibt = kuerzester_weg([orte[i] for i in indizes], [abstaende[i] for i in indizes])
    laenge += l
    anfang = [indizes[pos] for pos in bleibt]
    indizes = list(range(letzter, n))
    l, bleibt = kuerzester_weg([orte[i] for i in indizes], [abstaende[i] for i in indizes])
    laenge += l
    ende = [indizes[pos] for pos in bleibt]
    return laenge, anfang[:-1] + mitte + ende[1:]


laenge, neue_tour = kuerze_tour(orte, essentiell, abstaende, True)
print(laenge, neue_tour)

1020 [0, 1, 2, 5, 6, 7, 8, 9, 10, 15, 16, 17]


## Schritt 5: Ausgabe

Wir geben Länge, Anzahl der Tourpunkte und die Tourpunkte selbst aus (essentielle mit `X`), wie die Lösungshinweise.

In [ ]:
def zeige_tour(dateiname, startort_frei=True, tour_ausgeben=True):
    orte, jahre, essentiell, kumuliert = lies_tour(dateiname)
    abstaende = abstaende_berechnen(kumuliert)
    laenge, neue_tour = kuerze_tour(orte, essentiell, abstaende, startort_frei)
    print('Routenlänge:', laenge, '(davor', str(kumuliert[-1]) + ')')
    print('Anzahl Tourpunkte:', len(neue_tour), '(davor', str(len(orte)) + ')')
    if tour_ausgeben:
        for i in neue_tour:
            markierung = ' X' if essentiell[i] else ''
            print(orte[i], '(' + str(jahre[i]) + ')' + markierung)
    return laenge, neue_tour


_ = zeige_tour('beispieldaten/tour1.txt')

Routenlänge: 1020 (davor 2060)
Anzahl Tourpunkte: 12 (davor 18)
Brauerei (1613) X
Karzer (1665) X
Rathaus (1678) X
Rathaus (1739) X
Euler-Brücke (1768)
Fibonacci-Gaststätte (1820) X
Schiefes Haus (1823)
Theater (1880)
Emmy-Noether-Campus (1912) X
Emmy-Noether-Campus (1998) X
Euler-Brücke (1999)
Brauerei (2012)


Das stimmt mit dem Ergebnis der Lösungshinweise überein: Aus der Tour mit 2060 Metern wird eine mit 1020. Gestrichen werden der
Schlenker zwischen den beiden Rathaus-Besuchen (Gründungsstein, Wallanlage) und die Runde zwischen den beiden Besuchen des
Emmy-Noether-Campus (Fibonacci-Gaststätte 1923, Hilbert-Raum, Schiefes Haus 1950, Gauß-Turm).

## Alle Beispiele

Zu jedem Beispiel zeigen wir Länge und Anzahl, die vollständigen Touren stehen für `tour2` bis `tour5` in den Lösungshinweisen.
Hier geben wir die Tourpunkte nur für `tour3` aus. Bei `tour2` sind weniger Punkte essentiell als bei `tour1`
(Brauerei 1613 und Rathaus 1678 sind es nicht), die Länge bleibt aber 1020.

In [ ]:
for nr in range(1, 6):
    print('tour' + str(nr) + '.txt')
    _ = zeige_tour('beispieldaten/tour' + str(nr) + '.txt', True, nr == 3)
    print()

tour1.txt
Routenlänge: 1020 (davor 2060)
Anzahl Tourpunkte: 12 (davor 18)

tour2.txt
Routenlänge: 1020 (davor 2060)
Anzahl Tourpunkte: 12 (davor 18)

tour3.txt
Routenlänge: 2670 (davor 4560)
Anzahl Tourpunkte: 10 (davor 14)
Talstation (1768)
Wäldle (1805)
Mittlere Alp (1823)
Observatorium (1833)
Observatorium (1874) X
Piz Spitz (1898)
Panoramasteg (1912) X
Panoramasteg (1952)
Ziegenbrücke (1979) X
Talstation (2005)

tour4.txt
Routenlänge: 1420 (davor 3200)
Anzahl Tourpunkte: 16 (davor 29)

tour5.txt
Routenlänge: 2620 (davor 5000)
Anzahl Tourpunkte: 29 (davor 42)



### Startort nicht verändern (strenge Auslegung)

Bei `tour4` hängt das Ergebnis davon ab, ob der Startort wechseln darf. Mit freiem Startort wird die Tour von 3200 auf 1420 gekürzt,
mit festem Startort (`startort_frei=False`) nur auf 2000: Dann müssen Anfangsstück (Blaues Pferd, Alte Mühle) und Endstück (Baumschule,
Polizeipräsidium, Blaues Pferd) bleiben. Bei den anderen Beispielen macht es keinen Unterschied.

In [ ]:
for nr in range(1, 6):
    print('tour' + str(nr) + ': Startort fest, Länge', zeige_tour('beispieldaten/tour' + str(nr) + '.txt', False, False)[0])
print()
_ = zeige_tour('beispieldaten/tour4.txt', False)

Routenlänge: 1020 (davor 2060)
Anzahl Tourpunkte: 12 (davor 18)
tour1: Startort fest, Länge 1020
Routenlänge: 1020 (davor 2060)
Anzahl Tourpunkte: 12 (davor 18)
tour2: Startort fest, Länge 1020
Routenlänge: 2670 (davor 4560)
Anzahl Tourpunkte: 10 (davor 14)
tour3: Startort fest, Länge 2670
Routenlänge: 2000 (davor 3200)
Anzahl Tourpunkte: 21 (davor 29)
tour4: Startort fest, Länge 2000
Routenlänge: 2620 (davor 5000)
Anzahl Tourpunkte: 29 (davor 42)
tour5: Startort fest, Länge 2620

Routenlänge: 2000 (davor 3200)
Anzahl Tourpunkte: 21 (davor 29)
Blaues Pferd (1523)
Alte Mühle (1544)
Marktplatz (1549)
Marktplatz (1562)
Springbrunnen (1571)
Dom (1596) X
Bogenschütze (1610)
Bogenschütze (1683)
Schnecke (1698) X
Fischweiher (1710)
Reiterhof (1728) X
Schnecke (1742)
Schmiede (1765)
Große Gabel (1794)
Große Gabel (1874)
Fingerhut (1917) X
Stadion (1934)
Marktplatz (1962)
Baumschule (1974)
Polizeipräsidium (1991)
Blaues Pferd (2004)


## Alina ändert die essentiellen Tourpunkte

Das Programm bekommt die essentiellen Punkte als Liste von `True`/`False`, also kann Alina sie beliebig ändern. Beispiel: In `tour1` sollen
nur das Theater (Index 9) und die Brauerei 1613 (Index 0) essentiell sein. Die Länge ist zufällig wieder 1020, die Tour aber eine andere: Es entfällt der Schlenker zwischen den Rathaus-Besuchen, und nach dem Theater wird die Runde zwischen den beiden Besuchen des Emmy-Noether-Campus gestrichen.

In [ ]:
orte1, jahre1, _, kumuliert1 = lies_tour('beispieldaten/tour1.txt')
neu_essentiell = [False] * len(orte1)
neu_essentiell[0] = True
neu_essentiell[9] = True
laenge1, tour1_neu = kuerze_tour(orte1, neu_essentiell, abstaende_berechnen(kumuliert1), True)
print('Länge', laenge1, 'davor', kumuliert1[-1])
for i in tour1_neu:
    print(orte1[i], jahre1[i], 'X' if neu_essentiell[i] else '')

Länge 1020 davor 2060
Brauerei 1613 X
Karzer 1665 
Rathaus 1678 
Rathaus 1739 
Euler-Brücke 1768 
Fibonacci-Gaststätte 1820 
Schiefes Haus 1823 
Theater 1880 X
Emmy-Noether-Campus 1912 
Emmy-Noether-Campus 1998 
Euler-Brücke 1999 
Brauerei 2012 


## Zum Schluss: Aufwand und Hinweise

- **Aufwand:** Jeder Tourpunkt wird in `kuerzester_weg` genau einmal betrachtet (ein `dict` für die letzten Besuche), also reichen
  $O(n)$ Schritte für die ganze Tour.
- **Dijkstra:** Man kann das als kürzeste-Wege-Problem in einem Graphen sehen: Jeder Tourpunkt hat eine Kante zum Nachfolger und, falls
  der Ort schon vorkam, eine Kante mit Gewicht 0 vom früheren Besuch. Dijkstra liefert dasselbe, braucht aber $O(n \log n)$.
  Weil die Kanten alle nach vorne zeigen, genügt hier ein einziger Durchlauf von links nach rechts.
- **Warum nur der letzte Besuch?** Eine Kante vom drittletzten zum aktuellen Besuch ergibt sich aus den beiden Kanten dazwischen
  (jeweils Gewicht 0). Und weil bei Gleichstand der normale Weg gewählt wird, bleiben möglichst viele Tourpunkte erhalten.
- **Mehrere Lösungen:** Die kürzeste Strecke ist eindeutig, die Tour dazu nicht immer. Wir wählen bei Gleichstand die mit mehr Tourpunkten.

**MiniZinc:** Für diese Aufgabe lassen wir es weg. Das Verfahren ist ein einfacher Durchlauf in linearer Zeit; ein Modell wäre
umständlicher und langsamer als die Python-Lösung.